No,４

In [3]:
import os
import re
from pathlib import Path


# --- 共通の復元基盤 undo_utils.py を読み込む ---
# ※ このセルの import を並べ替えても壊れないように、undo_utils だけは
#    import 文ではなく importlib 経由で読み込んでいます。
import importlib
import sys
from pathlib import Path


def _locate_undo_utils():
    """undo_utils.py があるフォルダを探す（VS Code の作業ディレクトリ設定に依存しない）。"""
    candidates = []
    # 1) VS Code がノートブック自身のパスを教えてくれる場合
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        candidates.append(Path(nb_file).parent)
    # 2) 作業ディレクトリと、その中／親の「コードフォルダ」
    cwd = Path.cwd()
    candidates += [cwd, cwd / "コードフォルダ", cwd.parent, cwd.parent / "コードフォルダ"]

    for c in candidates:
        if (c / "undo_utils.py").is_file():
            return c.resolve()

    raise FileNotFoundError(
        "undo_utils.py が見つかりません。\n"
        "このノートブックと同じ「コードフォルダ」内に undo_utils.py があるか確認してください。\n"
        f"探した場所: {[str(c) for c in candidates]}"
    )


_uu_dir = str(_locate_undo_utils())
if _uu_dir not in sys.path:
    sys.path.insert(0, _uu_dir)

uu = importlib.import_module("undo_utils")
importlib.reload(uu)  # undo_utils.py を編集した場合も反映されるようにする

<module 'undo_utils' from 'C:\\Users\\0uh2j\\Desktop\\vscodeで\\ファイル整理２\\コードフォルダ\\undo_utils.py'>

拡張子ごとにフォルダ分けしたファイルの名前を「0+元のファイル名から抽出した連番数字_サブサブフォルダ名_サブフォルダ名」に変更

必ずセルごとに分けて実行する！

NR用

In [4]:
# === NR用【手順1】ファイル名を「0+連番_サブサブフォルダ名_サブフォルダ名」に変更 ===
# 変更内容は _undo/undo_log.json に記録され、末尾の復元セルで元に戻せます。

STEP_NAME = "04_ファイル名変更 (NR用 手順1 リネーム)"


def rename_csv_files_flexible():
    top_dir = uu.select_folder("一番上の大元フォルダを選択してください")
    if top_dir is None:
        return

    rename_count = 0

    with uu.UndoJournal(top_dir, STEP_NAME) as j:
        # uu.iter_files は _undo / _trash を除外して再帰的に走査する
        for file_path in uu.iter_files(top_dir, "*.csv"):
            original_name = file_path.name

            terminal_folder = file_path.parent.name             # 末端のフォルダ名
            pre_terminal_folder = file_path.parent.parent.name  # 末端の前のフォルダ名

            # 「$」の後の連番数字を抽出
            match = re.search(r"\$(\d+)\.csv$", original_name)
            if not match:
                continue

            seq_num = match.group(1)
            new_name = f"0{seq_num}_{pre_terminal_folder}_{terminal_folder}.csv"

            try:
                j.move(file_path, file_path.parent / new_name)
                print(f"変更完了: {pre_terminal_folder}/{terminal_folder}/{original_name} -> {new_name}")
                rename_count += 1
            except Exception as e:
                print(f"エラー ({original_name}): {e}")

    print()
    print(f"処理が完了しました。合計 {rename_count} 件のファイル名を変更しました。")


rename_csv_files_flexible()

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-直接式樹脂圧力本実験データ再々/1.0～2.0mm/csv
変更完了: 190℃/010/010$10.csv -> 010_190℃_010.csv
変更完了: 190℃/010/010$11.csv -> 011_190℃_010.csv
変更完了: 190℃/010/010$12.csv -> 012_190℃_010.csv
変更完了: 190℃/010/010$3.csv -> 03_190℃_010.csv
変更完了: 190℃/010/010$4.csv -> 04_190℃_010.csv
変更完了: 190℃/010/010$5.csv -> 05_190℃_010.csv
変更完了: 190℃/010/010$6.csv -> 06_190℃_010.csv
変更完了: 190℃/010/010$7.csv -> 07_190℃_010.csv
変更完了: 190℃/010/010$8.csv -> 08_190℃_010.csv
変更完了: 190℃/010/010$9.csv -> 09_190℃_010.csv
変更完了: 190℃/020/020$10.csv -> 010_190℃_020.csv
変更完了: 190℃/020/020$11.csv -> 011_190℃_020.csv
変更完了: 190℃/020/020$12.csv -> 012_190℃_020.csv
変更完了: 190℃/020/020$3.csv -> 03_190℃_020.csv
変更完了: 190℃/020/020$4.csv -> 04_190℃_020.csv
変更完了: 190℃/020/020$5.csv -> 05_190℃_020.csv
変更完了: 190℃/020/020$6.csv -> 06_190℃_020.csv
変更完了: 190℃/020/020$7.csv -> 07_190℃_020.csv
変更完了: 190℃/020/020$8.csv -> 08_190℃_020.csv
変更完了: 190℃/020/020$9

ファイル名変更後のファイルの移動と、ファイルが入っていた末端フォルダの削除

In [5]:
# === NR用【手順2】末端フォルダの削除・前フォルダへの移動・連番の振り直し ===
# 2段階リネーム（temp_ を経由）の各 rename を1件ずつ記録するので、
# 復元セルは逆順に再生するだけで正しく元の名前に戻せます。

STEP_NAME = "04_ファイル名変更 (NR用 手順2 移動と連番振り直し)"


def reorganize_resort_and_rename_001():
    top_dir = uu.select_folder("一番上の大元フォルダを選択してください")
    if top_dir is None:
        return

    pre_term_files = {}
    folders_to_delete = set()

    with uu.UndoJournal(top_dir, STEP_NAME) as j:
        # ------------------------------------------
        # 手順2-1: ファイルを末端の1つ上のフォルダへ移動
        # ------------------------------------------
        for file_path in list(uu.iter_files(top_dir, "*.csv")):
            terminal_folder = file_path.parent
            pre_terminal_folder = file_path.parent.parent

            if terminal_folder == top_dir or pre_terminal_folder == top_dir.parent:
                continue

            try:
                temp_move_path = pre_terminal_folder / f"temp_{file_path.name}"
                j.move(file_path, temp_move_path)

                folders_to_delete.add(terminal_folder)
                pre_term_files.setdefault(pre_terminal_folder, []).append(temp_move_path)
            except Exception as e:
                print(f"移動エラー ({file_path.name}): {e}")

        # ------------------------------------------
        # 手順2-2: 後ろの数字でソートし、001 から連番を振り直す
        # ------------------------------------------
        for pre_term_dir, files in pre_term_files.items():
            sortable_list = []

            for temp_path in files:
                original_name = temp_path.name.replace("temp_", "", 1)
                match = re.match(r"^(\d+)_(.*)_(\d+)\.csv$", original_name)

                if match:
                    sortable_list.append({
                        "front_num": int(match.group(1)),
                        "middle_str": match.group(2),
                        "back_str": match.group(3),
                        "back_num": int(match.group(3)),
                        "temp_path": temp_path,
                    })
                else:
                    print(f"警告: ファイル名形式が一致しません。リネームをスキップします: {original_name}")
                    j.move(temp_path, pre_term_dir / original_name)

            sortable_list.sort(key=lambda x: (x["back_num"], x["front_num"]))

            # 1 からスタートし、3桁のゼロ埋め（001, 002, ...）にする
            for i, item in enumerate(sortable_list, start=1):
                new_name = f"{i:03d}_{item['middle_str']}_{item['back_str']}.csv"
                try:
                    j.move(item["temp_path"], pre_term_dir / new_name)
                    print(f"整理完了: {pre_term_dir.name}/{new_name}")
                except Exception as e:
                    print(f"リネームエラー: {e}")

        # ------------------------------------------
        # 手順2-3: 空になった末端フォルダの削除
        # ------------------------------------------
        print()
        print("--- フォルダの整理を開始します ---")
        deleted_folder_count = 0

        for folder in sorted(folders_to_delete):
            # .DS_Store などが残っていても削除せず _trash へ退避する
            for leftover in sorted(folder.iterdir()):
                if leftover.is_file():
                    j.trash(leftover)

            if j.rmdir(folder):
                deleted_folder_count += 1
            else:
                print(f"スキップ: {folder.name}（中身が残っているため保持しました）")

    print()
    print("すべての処理が完了しました！")
    print(f"{deleted_folder_count} 個の不要な末端フォルダを削除しました。")


reorganize_resort_and_rename_001()

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-直接式樹脂圧力本実験データ再々/1.0～2.0mm/csv
整理完了: 190℃/001_190℃_010.csv
整理完了: 190℃/002_190℃_010.csv
整理完了: 190℃/003_190℃_010.csv
整理完了: 190℃/004_190℃_010.csv
整理完了: 190℃/005_190℃_010.csv
整理完了: 190℃/006_190℃_010.csv
整理完了: 190℃/007_190℃_010.csv
整理完了: 190℃/008_190℃_010.csv
整理完了: 190℃/009_190℃_010.csv
整理完了: 190℃/010_190℃_010.csv
整理完了: 190℃/011_190℃_020.csv
整理完了: 190℃/012_190℃_020.csv
整理完了: 190℃/013_190℃_020.csv
整理完了: 190℃/014_190℃_020.csv
整理完了: 190℃/015_190℃_020.csv
整理完了: 190℃/016_190℃_020.csv
整理完了: 190℃/017_190℃_020.csv
整理完了: 190℃/018_190℃_020.csv
整理完了: 190℃/019_190℃_020.csv
整理完了: 190℃/020_190℃_020.csv
整理完了: 190℃/021_190℃_040.csv
整理完了: 190℃/022_190℃_040.csv
整理完了: 190℃/023_190℃_040.csv
整理完了: 190℃/024_190℃_040.csv
整理完了: 190℃/025_190℃_040.csv
整理完了: 190℃/026_190℃_040.csv
整理完了: 190℃/027_190℃_040.csv
整理完了: 190℃/028_190℃_040.csv
整理完了: 190℃/029_190℃_040.csv
整理完了: 190℃/030_190℃_040.csv
整理完了: 190℃/031_190℃_080.csv


Futaba用

In [ ]:
# === Futaba用【セル1】更新時刻順にソートしてリネーム ===
# 変更内容は _undo/undo_log.json に記録されます。
# 以前あった「既に undo_log.json が存在します」ガードは不要になりました
# （履歴はスタックとして積まれるので、何度実行しても1工程ずつ戻せます）。

STEP_NAME = "04_ファイル名変更 (Futaba用 セル1 リネーム)"


def step1_extract_sort_rename():
    top_dir = uu.select_folder("大元のメインフォルダを選択")
    if top_dir is None:
        return

    target_groups = {}

    # 1. ファイルの走査とグループ化（_undo / _trash は除外される）
    for file_path in uu.iter_files(top_dir, "*.csv"):
        terminal_folder = file_path.parent
        pre_terminal_folder = terminal_folder.parent

        if terminal_folder == top_dir or pre_terminal_folder == top_dir.parent:
            continue

        target_groups.setdefault(pre_terminal_folder, []).append({
            "path": file_path,
            "terminal_name": terminal_folder.name,
            "pre_terminal_name": pre_terminal_folder.name,
            "mtime": file_path.stat().st_mtime,
        })

    if not target_groups:
        print("処理対象のCSVファイルが見つかりませんでした。")
        return

    rename_count = 0

    # 2. 時系列ソートとリネームの実行
    with uu.UndoJournal(top_dir, STEP_NAME) as j:
        for pre_term, files_info in target_groups.items():
            files_info.sort(key=lambda x: x["mtime"])

            for i, info in enumerate(files_info, start=1):
                new_name = f"{i:03d}_{info['pre_terminal_name']}_{info['terminal_name']}.csv"
                old_path = info["path"]

                try:
                    j.move(old_path, old_path.parent / new_name)
                    rename_count += 1
                except Exception as e:
                    print(f"エラー ({old_path.name}): {e}")

    print(f"【完了】合計 {rename_count} 件のファイルをリネームしました。")
    print("問題がなければ、続いてセル2を実行してください。")


step1_extract_sort_rename()

In [ ]:
# === Futaba用【セル2】1つ上の階層へ移動し、空になった末端フォルダを削除 ===

STEP_NAME = "04_ファイル名変更 (Futaba用 セル2 移動と空フォルダ削除)"


def step2_move_and_delete():
    top_dir = uu.select_folder("大元のメインフォルダを選択")
    if top_dir is None:
        return

    move_count = 0
    deleted_folder_count = 0
    folders_to_delete = set()

    with uu.UndoJournal(top_dir, STEP_NAME) as j:
        # 1. ファイルを1つ上の階層へ移動
        for current_path in list(uu.iter_files(top_dir, "*.csv")):
            terminal_folder = current_path.parent
            pre_terminal_folder = terminal_folder.parent

            if terminal_folder == top_dir or pre_terminal_folder == top_dir.parent:
                continue

            try:
                j.move(current_path, pre_terminal_folder / current_path.name)
                folders_to_delete.add(terminal_folder)
                move_count += 1
            except Exception as e:
                print(f"移動エラー ({current_path.name}): {e}")

        # 2. 空になったフォルダの削除
        for folder in sorted(folders_to_delete):
            # .DS_Store などが残っていても削除せず _trash へ退避する
            for leftover in sorted(folder.iterdir()):
                if leftover.is_file():
                    j.trash(leftover)

            if j.rmdir(folder):
                deleted_folder_count += 1

    print(f"【完了】合計 {move_count} 件のファイルを移動し、"
          f"{deleted_folder_count} 個の空フォルダを削除しました。")


step2_move_and_delete()

---
### ⏪ 復元（元に戻す）

このセルを実行すると、**このノートブックで行った直前の1工程**を巻き戻します。
（メインフォルダの `_undo/undo_log.json` に記録された履歴を使います）

繰り返し実行すれば、01〜06 のどの工程まででもさかのぼれます。
削除したファイルは `_trash` フォルダに退避されているので、これも一緒に元の場所へ戻ります。

In [ ]:
# ===== 共通の復元セル =====
# 直前に実行した1工程を巻き戻します。
# 続けて実行すれば、さらに1つ前の工程へとさかのぼれます。

uu.undo_interactive()